# L2：构建 Memory Manager（记忆管理器）

<div style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> <p>⏳ <b>注意 <code>（数据库启动中）</code>：</b>本 notebook 大约需要 30-60 秒才能就绪。等待期间可以先开始观看视频。</p>
<p>如果运行第一个单元格后看到 <tt>Admin connection failed</tt>，只需稍等片刻再重新运行——这不是凭证问题。</p>
</div>

本课介绍 AI Agent 的**结构化记忆**原则：不同的智能体记忆类型需要各自不同的数据模型、索引策略和检索方法，并通过一个统一的 Memory Manager（记忆管理器）来协调。学完本课，你将能够为核心智能体记忆类型设计并实现持久化记忆存储、为高效检索建模记忆数据，并构建一个记忆管理器来编排智能体在执行过程中如何存储、检索和操作记忆。

本 lab 贯穿始终的用例是一个**智能体式研究助理（agentic research assistant）**，帮助用户跨多个会话研究复杂主题。助理必须记住先前的发现、信息来源可信度以及用户偏好，从而给出一致的、上下文感知的回答，而不必每次都重复相同的探索工作。

**本课目标**

学完本课，你将理解如何：
- 解释核心智能体记忆类型，以及它们在构建可靠、长时间运行的智能体系统中的作用。
- 设计持久化的智能体记忆架构，把各记忆类型映射到合适的存储后端（SQL 表和向量存储）。
- 用 Oracle Vector Search 实现语义记忆，包括嵌入（embeddings）、OracleVS 配置、HNSW 索引和 metadata 过滤。
- 构建一个记忆管理器，编排智能体在执行过程中如何存储、检索和更新记忆。
- 评估记忆设计的取舍，在检索准确率、延迟、成本和智能体可靠性之间做权衡。


本节演示如何使用 **LangChain 的 Oracle Vector Store（OracleVS）**，基于语义相似度来存储和检索文档。

向量检索让我们能按**含义**查找文档，而不是靠精确的关键词匹配。

## 你将学到什么

| 步骤 | 说明 |
|------|-------------|
| **1. 初始化嵌入模型** | 加载 HuggingFace 嵌入模型，把文本转成向量 |
| **2. 创建向量存储** | 建立基于 Oracle 的向量存储，并指定距离策略 |
| **3. 创建索引** | 构建 HNSW 索引，加速相似度检索 |
| **4. 写入文档** | 把文本连同 metadata 一起存入向量数据库 |
| **5. 查询** | 用自然语言检索相似文档 |
| **6. 过滤结果** | 用 metadata 过滤器缩小检索范围 |

**关键组件**

- **`OracleVS`**：LangChain 的 Oracle 向量存储集成
- **`HuggingFaceEmbeddings`**：把文本转成 768 维向量
- **`DistanceStrategy.EUCLIDEAN_DISTANCE`**：用欧氏距离度量向量间相似度
- **HNSW 索引**：基于图的近邻遍历，加速相似度检索


## Part 1：搭建数据库、向量存储与嵌入模型

In [ ]:
from helper import suppress_warnings

# 警告控制
suppress_warnings()

from helper import load_env, setup_oracle_database, connect_to_oracle

load_env()

# 一次性管理员初始化：配置表空间、向量内存和 VECTOR 用户
setup_oracle_database()

# 后续所有操作都以 VECTOR 用户身份连接
database_connection = connect_to_oracle(
    user="VECTOR",
    password="VectorPwd_2025",
    dsn="127.0.0.1:1521/FREEPDB1",
    program="devrel.deeplearning.course_1",
)

print("Using user:", database_connection.username)

### 加载嵌入模型

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# 初始化嵌入模型
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

### 定义记忆表与存储
首先，为每种记忆类型定义表名。

这些表将创建在 Oracle 数据库中，用于持久化智能体记忆。

### 我们要实现的记忆类型

| 记忆类型 | 人类类比 | 用途 | 存储 | 使用的检索策略 |
|-------------|---------------|---------|---------|---------------------------|
| **对话记忆（Conversational）** | 短期记忆 | 每个线程的聊天历史 | SQL 表 | 按 thread_id 精确匹配 |
| **知识库（Knowledge Base）** | 长期语义记忆 | 事实、文档、搜索结果 | 向量存储 | 语义相似度检索 |
| **工作流（Workflow）** | 程序性记忆 | 学到的行动模式 | 向量存储 | 语义相似度检索 + metadata 过滤 |
| **工具箱（Toolbox）** | 技能记忆 | 可用工具与能力 | 向量存储 | 语义相似度检索 |
| **实体（Entity）** | 情景记忆 | 提到的人、地点、系统 | 向量存储 | 语义相似度检索 |
| **摘要（Summary）** | 压缩记忆 | 长对话的浓缩上下文 | 向量存储 | 语义相似度检索（可选按 ID 过滤） |
| **工具日志（Tool Log）** | 执行审计轨迹 | 工具的原始输入/输出与执行状态 | SQL 表 | 按 thread_id 精确匹配 + 按时间戳排序 |


In [ ]:
# 每种记忆类型对应的表名
CONVERSATIONAL_TABLE   = "CONVERSATIONAL_MEMORY" # 情景记忆（Episodic）
KNOWLEDGE_BASE_TABLE   = "SEMANTIC_MEMORY" # 语义记忆（Semantic）
WORKFLOW_TABLE = "WORKFLOW_MEMORY" # 程序性记忆（Procedural）
TOOLBOX_TABLE    = "TOOLBOX_MEMORY" # 程序性记忆（Procedural）
ENTITY_TABLE = "ENTITY_MEMORY" # 语义记忆（Semantic）
SUMMARY_TABLE = "SUMMARY_MEMORY" # 语义记忆（Semantic）
TOOL_LOG_TABLE = "TOOL_LOG_MEMORY" # 工具执行日志

ALL_TABLES = [
    CONVERSATIONAL_TABLE,
    KNOWLEDGE_BASE_TABLE,
    WORKFLOW_TABLE,
    TOOLBOX_TABLE,
    ENTITY_TABLE,
    SUMMARY_TABLE,
    TOOL_LOG_TABLE]

# 删除已存在的表，从头开始
for table in ALL_TABLES:
    try:
        with database_connection.cursor() as cur:
            cur.execute(f"DROP TABLE {table} PURGE")
            print(f"  - {table} (dropped)")
    except Exception as e:
        if "ORA-00942" in str(e):
            print(f"  - {table} (not exists)")
        else:
            print(f"  ✗ {table}: {e}")

database_connection.commit()

### 创建对话记忆表

下面这个函数创建一个存储聊天历史的 SQL 表。

与向量存储不同，对话记忆使用传统的表，因为我们需要按 thread ID **精确检索**（而不是相似度检索）。

**它做了什么：**
- 创建一个包含 `id`、`thread_id`、`role`、`content`、`timestamp`、`metadata` 等列的表
- 在 `thread_id` 上建索引，加速会话查找
- 在 `timestamp` 上建索引，用于按时间排序


In [ ]:
def create_conversational_history_table(conn, table_name: str = "CONVERSATIONAL_MEMORY"):
    """
    创建用于存储对话历史的表。

    参数:
        conn: Oracle 数据库连接
        table_name: 要创建的表名
    """
    with conn.cursor() as cur:
        # 如果表已存在则先删除
        try:
            cur.execute(f"DROP TABLE {table_name}")
        except:
            pass  # 表不存在

        # 按正确的 schema 建表
        cur.execute(f"""
            CREATE TABLE {table_name} (
                id VARCHAR2(100) DEFAULT SYS_GUID() PRIMARY KEY,
                thread_id VARCHAR2(100) NOT NULL,
                role VARCHAR2(50) NOT NULL,
                content CLOB NOT NULL,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                metadata CLOB,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                summary_id VARCHAR2(100) DEFAULT NULL
            )
        """)

        # 在 thread_id 上建索引，加速查找
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_thread_id ON {table_name}(thread_id)
        """)

        # 在 timestamp 上建索引，用于排序
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_timestamp ON {table_name}(timestamp)
        """)

    conn.commit()
    print(f"Table {table_name} created successfully with indexes")
    return table_name


In [ ]:
from helper import create_tool_log_table

# 创建 SQL 记忆表
CONVERSATION_HISTORY_TABLE = create_conversational_history_table(database_connection, CONVERSATIONAL_TABLE)
TOOL_LOG_HISTORY_TABLE = create_tool_log_table(database_connection, TOOL_LOG_TABLE)

### 为每种记忆类型创建向量存储

这里我们创建 5 个相互独立的向量存储——每种记忆类型一个。

每个向量存储背后是各自独立的 Oracle 表，并且为了一致性都使用同一个嵌入模型。

| 向量存储 | 用途 |
|--------------|---------|
| `knowledge_base_vs` | 存储文档、事实和搜索结果 |
| `workflow_vs` | 存储学到的行动模式和工具调用序列 |
| `toolbox_vs` | 存储工具定义，支持语义化工具发现 |
| `entity_vs` | 存储抽取出的实体（人、地点、系统） |
| `summary_vs` | 存储长对话的压缩摘要 |


In [ ]:
from langchain_oracledb.vectorstores import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_oracledb.retrievers.hybrid_search import (
    OracleVectorizerPreference
)


class StoreManager:
    """统一管理所有存储（向量存储和 SQL 表），通过 getter 方法方便地获取。"""

    def __init__(self, client, embedding_function, table_names, distance_strategy, conversational_table, tool_log_table: str | None = None):
        """
        初始化所有存储。

        参数:
            client: Oracle 数据库连接
            embedding_function: 使用的嵌入模型
            table_names: 字典，键为 knowledge_base、workflow、toolbox、entity、summary
            distance_strategy: 向量检索的距离策略
            conversational_table: 对话历史 SQL 表的表名
            tool_log_table: 工具日志 SQL 表的表名
        """
        self.client = client
        self.embedding_function = embedding_function
        self.distance_strategy = distance_strategy
        self._conversational_table = conversational_table
        self._tool_log_table = tool_log_table

        # 初始化所有向量存储
        self._knowledge_base_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['knowledge_base'],
            distance_strategy=distance_strategy,
        )

        self._workflow_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['workflow'],
            distance_strategy=distance_strategy,
        )

        self._toolbox_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['toolbox'],
            distance_strategy=distance_strategy,
        )

        self._entity_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['entity'],
            distance_strategy=distance_strategy,
        )

        self._summary_vs = OracleVS(
            client=client,
            embedding_function=embedding_function,
            table_name=table_names['summary'],
            distance_strategy=distance_strategy,
        )

        # 保存知识库的混合检索偏好设置（可选）
        self._kb_vectorizer_pref = None

    def get_knowledge_base_store(self):
        """返回知识库向量存储。"""
        return self._knowledge_base_vs

    def get_workflow_store(self):
        """返回工作流向量存储。"""
        return self._workflow_vs

    def get_toolbox_store(self):
        """返回工具箱向量存储。"""
        return self._toolbox_vs

    def get_entity_store(self):
        """返回实体向量存储。"""
        return self._entity_vs

    def get_summary_store(self):
        """返回摘要向量存储。"""
        return self._summary_vs

    def get_conversational_table(self):
        """返回对话历史表的表名。"""
        return self._conversational_table


    def get_tool_log_table(self):
        """返回工具日志表的表名。"""
        return self._tool_log_table

    def setup_hybrid_search(self, preference_name="KB_VECTORIZER_PREF"):
        """
        为知识库设置混合检索（hybrid search）。
        创建用于混合索引的 vectorizer preference。
        """
        self._kb_vectorizer_pref = OracleVectorizerPreference.create_preference(
            vector_store=self._knowledge_base_vs,
            preference_name=preference_name
        )
        return self._kb_vectorizer_pref


In [ ]:
# 创建 StoreManager 实例
store_manager = StoreManager(
    client=database_connection,
    embedding_function=embedding_model,
    table_names={
        'knowledge_base': KNOWLEDGE_BASE_TABLE,
        'workflow': WORKFLOW_TABLE,
        'toolbox': TOOLBOX_TABLE,
        'entity': ENTITY_TABLE,
        'summary': SUMMARY_TABLE,
    },
    distance_strategy=DistanceStrategy.COSINE,
    conversational_table=CONVERSATION_HISTORY_TABLE,
    tool_log_table=TOOL_LOG_HISTORY_TABLE,
)

In [ ]:
# 通过 manager 获取所有存储
conversation_table = store_manager.get_conversational_table()
knowledge_base_vs = store_manager.get_knowledge_base_store()
workflow_vs = store_manager.get_workflow_store()
toolbox_vs = store_manager.get_toolbox_store()
entity_vs = store_manager.get_entity_store()
summary_vs = store_manager.get_summary_store()
tool_log_table = store_manager.get_tool_log_table()

In [ ]:
from helper import safe_create_index

print("Creating vector indexes...")
safe_create_index(database_connection, knowledge_base_vs, "knowledge_base_vs_ivf")
safe_create_index(database_connection, workflow_vs, "workflow_vs_ivf")
safe_create_index(database_connection, toolbox_vs, "toolbox_vs_ivf")
safe_create_index(database_connection, entity_vs, "entity_vs_ivf")
safe_create_index(database_connection, summary_vs, "summary_vs_ivf")
print("All indexes created!")

### 智能体系统中记忆操作的分类

记忆工程中的一个关键设计决策：哪些操作应该是**确定性的（Deterministic）**（由代码自动执行），哪些应该是**智能体触发的（Agent-Triggered）**（由 LLM 在运行时决定）。

- **确定性**记忆操作基于系统规则运行，而不由模型自行裁量。它每次都执行（或在明确定义、不可协商的条件下执行），从而让系统行为可预测。
- **智能体触发**的记忆操作只在模型基于意图和情境判断有必要时才运行。

| 操作 | 确定性 | 智能体触发 |
|-----------|:------------:|:-------:|
| `read_conversational_memory()` | ✅ | ❌ |
| `read_knowledge_base()` | ✅ | ❌ |
| `read_workflow()` | ✅ | ❌ |
| `read_entity()` | ✅ | ❌ |
| `read_summary_context()` | ❌ | ✅ |
| `write_conversational_memory()` | ✅ | ❌ |
| `write_workflow()` | ✅ | ❌ |
| `write_entity()` | ❌ | ✅ |
| `search_tavily()` | ❌ | ✅ |
| `expand_summary()` | ❌ | ✅ |
| `summarize_and_store()` | ❌ | ✅ |
| `read_toolbox()` | ✅ | ✅ |


确定性记忆操作的运行时机：
- **每一轮都运行**，或
- 在**明确、固定的条件**下运行（例如"总是在 agent loop 开始时"、"总是在工具执行之后"）

### 为什么确定性检索有用
记忆检索通常放在**每次 agent loop 开始时**运行，因为：

1. **上下文引导（context bootstrapping）不可协商**
   - 智能体需要先前的上下文才能保持一致、避免重复犯错。
   - 没有确定性检索，智能体就表现得"无状态"，每次从零开始。

2. **智能体无法主动去查它不知道存在的东西**
   - 如果让智能体自己决定要不要查记忆，它就得先猜里面存了什么。
   - 这造成鸡生蛋问题：*你需要记忆才能知道自己需要哪些记忆。*

3. **可预测性**
   - 总是加载记忆能产生一致的行为，让系统更容易评估和调试。

### 为什么确定性存储有用
持久化对话、工作流和实体通常是确定性的，因为：

1. **可靠性**
   - 你不希望智能体"忘了保存"重要信息。
   - 如果连续性很重要，持久化就必须保持一致。

2. **完整性**
   - 每次交互都应被记录，避免出现空缺。
   - 选择性保存会造成上下文缺失，之后会拖垮长程任务。

3. **降低认知负担**
   - 模型应该专注于任务执行，而不是记忆的簿记工作。

### 确定性记忆操作的优势
- 跨运行、跨轮次的**行为可预测**
- **更强的连续性**（更少"无状态重置"）
- **更少漏存记忆**（更高可靠性）
- **更容易调试和评估**（对该加载/保存什么有明确预期）


这适用于需要判断力的记忆动作，例如：
- "这条信息应该保存为持久偏好吗？"
- "现在应该做整合/总结吗？"
- "在基线预加载之外，我还需要更深入的检索吗？"
- "我应该强化、更新、合并还是衰减这条记忆？"

### 为什么智能体触发的记忆操作有用

1. **相关性**
   - 并非所有内容都值得长期存储。
   - 智能体能区分信号（偏好、决策、约束）和噪声。

2. **成本与延迟控制**
   - 深度检索、重排（reranking）、总结和整合都要消耗 token/时间。
   - 只在需要时触发能降低开销。

3. **更高质量的记忆管理**
   - 决定*存什么*、*怎么压缩*需要对意图的语义理解。
   - 模型很适合判断某个记忆动作是否值得执行。

### 智能体触发记忆操作的优势
- **信噪比更高的记忆**（更少杂物）
- **减少记忆膨胀**
- **按需使用计算**（只在有价值时才总结/展开/检索）
- **更接近人类的记忆方式**（在重要时刻存取）

### 工具调用如何融入这个框架

外部工具调用（如网页搜索、外部数据库查询、昂贵的总结任务）通常是**智能体触发**的，因为：

1. **意图很重要**
   - 只有智能体能判断是否需要额外信息。
   - 对每个查询都自动调用工具是浪费。

2. **成本考量**
   - 工具往往引入延迟，还可能产生 API 费用。
   - 智能体应只在预期价值高时才调用工具。

3. **需要判断力**
   - 决定*搜什么*、*展开什么*需要理解用户的目标。

---

## Part 2：初始化 Memory Manager

`MemoryManager` 类是统一所有记忆操作的核心抽象。它为不同记忆类型的读写提供干净的接口，隐藏了 SQL 查询和向量存储操作的复杂性。这是一个用一致的读/写模式管理 7 种记忆类型的单一类：

| 记忆类型 | 存储 | 写方法 | 读方法 |
|-------------|---------|--------------|-------------|
| **对话记忆** | SQL 表 | `write_conversational_memory()` | `read_conversational_memory()` |
| **知识库** | 向量存储 | `write_knowledge_base()` | `read_knowledge_base()` |
| **工作流** | 向量存储 | `write_workflow()` | `read_workflow()` |
| **工具箱** | 向量存储 | `write_toolbox()` | `read_toolbox()` |
| **实体** | 向量存储 | `write_entity()` | `read_entity()` |
| **摘要** | 向量存储 | `write_summary()` | `read_summary_memory()`、`read_summary_context()` |
| **工具日志** | SQL 表 | `write_tool_log()` | `read_tool_logs()` |


In [ ]:
from helper import MemoryManager

# 初始化 MemoryManager 实例
# 注意：对话记忆用 SQL 表，其余记忆类型用向量存储
memory_manager = MemoryManager(
    conn=database_connection,
    conversation_table=CONVERSATION_HISTORY_TABLE,
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs,
    tool_log_table=TOOL_LOG_HISTORY_TABLE
)

## Part 3：使用 Memory Manager

为 **ArxivScout** 灌入知识库：从 HuggingFace 以流式（streaming）方式读取 `nick007x/arxiv-papers` 数据集中的 arXiv 论文记录。

In [ ]:
from datasets import load_dataset
from itertools import islice

ds = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

只提取关键字段（标题、学科、摘要、提交日期和 arXiv ID），把标题 + 学科 + 摘要拼接成一段可检索的文本，然后通过 `memory_manager.write_knowledge_base(...)` 写入，把提取的字段作为 metadata 一并存储，用于过滤和溯源。

In [ ]:
for paper in islice(ds, 100):
    # 提取关键字段
    title = (paper.get("title") or "").strip()
    abstract = (paper.get("abstract") or "").strip()
    subjects = (paper.get("subjects") or paper.get("primary_subject") or "").strip()
    submission_date = (paper.get("submission_date") or "").strip()

    # 跳过空记录
    if not (title or abstract or subjects):
        continue

    # 把包含上下文的关键字段拼接起来，用于语义检索
    text = "\n".join([part for part in (title, subjects, abstract) if part])

    memory_manager.write_knowledge_base(
        text=text,
        metadata={
            "arxiv_id": paper.get("arxiv_id"),
            "title": title,
            "subjects": subjects,
            "abstract": abstract,
            "submission_date": submission_date,
        },
    )

In [ ]:
results = memory_manager.read_knowledge_base(query="space exploration")
print(results)